# Local Qwen Evaluation Notebook

This notebook is an interactive version of `scripts/evaluation/run_local_qwen_eval.py`.

Run cells top-to-bottom.

In [ ]:
import subprocess
import sys


# Clean + pinned install to avoid mixed transformers files in shared runtimes.
PIP_UNINSTALL = [
    "transformers",
    "tokenizers",
    "huggingface-hub",
    "accelerate",
]
PIP_INSTALL = [
    "torch==2.4.1",
    "transformers==4.45.2",
    "tokenizers==0.20.1",
    "huggingface-hub==0.25.2",
    "accelerate==0.34.2",
    "tqdm",
    "sacrebleu",
    "sentencepiece",
    "ipywidgets",
]


def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


def clean_reinstall_runtime_packages() -> None:
    # Uninstall first; ignore errors if a package is not currently installed.
    run([sys.executable, "-m", "pip", "uninstall", "-y", *PIP_UNINSTALL])

    # Reinstall exact compatible versions.
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--upgrade", *PIP_INSTALL])

    # Verify versions in this environment.
    import torch
    import transformers

    print(f"Verified torch: {torch.__version__}")
    print(f"Verified transformers: {transformers.__version__}")
    print("Done. Restart kernel once, then run notebook from the imports cell.")


clean_reinstall_runtime_packages()

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

In [ ]:
import re
from collections import Counter, defaultdict

# Flat copy mode: notebook and data files live in the same directory.
RUN_DIR = Path.cwd().resolve()
print('RUN_DIR:', RUN_DIR)

TAG_BY_LANGUAGE = {
    'german': 'deu',
    'de': 'deu',
    'deu': 'deu',
    'spanish': 'es',
    'es': 'es',
    'russian': 'ru',
    'ru': 'ru',
}


def load_jsonl(path: str | os.PathLike[str]) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def save_jsonl(path: str | os.PathLike[str], records: list[dict[str, Any]]) -> None:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')


def terminology_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str] | None:
    if mode == 'no_term':
        return None
    if mode == 'proper_term':
        terminology = (sample.get('proper_terms') or {}).copy()
        return terminology or None
    if mode == 'random_term':
        terminology = (sample.get('random_terms') or {}).copy()
        for key in (sample.get('proper_terms') or {}).keys():
            terminology.pop(key, None)
        return terminology or None
    return None


def output_tag_for_language(target_lang: str) -> str:
    return TAG_BY_LANGUAGE.get(str(target_lang).strip().lower(), 'deu')


def strip_output_tags(text: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(
        r'</?(deu|de|ger|german|es|spa|spanish|ru|rus|russian)>',
        '',
        text,
        flags=re.IGNORECASE,
    ).strip()


def build_translation_prompt(
    source_text: str,
    target_lang: str,
    terminology: dict[str, str] | None = None,
) -> str:
    output_tag = output_tag_for_language(target_lang)
    term_block = ''

    if terminology:
        term_block = 'Terminology:\n'
        for src, tgt in terminology.items():
            term_block += f'{src} -> {tgt}\n'
        term_block += '\n'

    return f"""
Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to {target_lang}.

{term_block}
Input:
<en> {source_text} </en>
"""


def compute_bleu_chrf(hyps: list[str], refs: list[str]) -> dict[str, float]:
    import sacrebleu

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {
        'bleu': bleu.score,
        'chrf': chrf.score,
    }


def _normalize_text(text: str) -> str:
    return ' '.join(str(text).lower().split())


def _count_term_occurrences(term: str, text: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    pattern = r'\b' + re.escape(term_norm) + r'\b'
    return len(re.findall(pattern, text_norm))


def terminology_accuracy_advanced(
    samples: list[dict[str, Any]],
    predictions: list[str],
    mode: str = 'proper_term',
) -> dict[str, Any]:
    term_ratios: dict[str, float] = {}
    total_terms = 0

    for sample, pred in zip(samples, predictions):
        if mode == 'proper_term':
            terms = (sample.get('proper_terms') or {}).copy()
        elif mode == 'random_term':
            terms = (sample.get('random_terms') or {}).copy()
            for key in (sample.get('proper_terms') or {}).keys():
                terms.pop(key, None)
        else:
            terms = {}

        source_text = sample.get('en', '')

        for src, tgt in terms.items():
            total_terms += 1

            src_count = _count_term_occurrences(src, source_text)
            if src_count == 0:
                src_count = 1

            tgt_count = _count_term_occurrences(tgt, pred)
            ratio = min(tgt_count / src_count, 1.0)

            term_ratios[src] = ratio

    avg_accuracy = (
        sum(term_ratios.values()) / len(term_ratios) * 100
        if term_ratios
        else None
    )

    return {
        'total_terms': total_terms,
        'avg_ratio_pct': avg_accuracy,
        'per_term_ratios': term_ratios,
    }


def terminology_consistency_advanced(
    samples: list[dict[str, Any]],
    predictions: list[str],
    mode: str = 'proper_term',
) -> dict[str, Any]:
    term_to_candidates: dict[str, list[str]] = defaultdict(list)

    for sample, pred in zip(samples, predictions):
        if mode == 'proper_term':
            terms = (sample.get('proper_terms') or {}).copy()
        elif mode == 'random_term':
            terms = (sample.get('random_terms') or {}).copy()
            for key in (sample.get('proper_terms') or {}).keys():
                terms.pop(key, None)
        else:
            terms = {}

        for src, tgt in terms.items():
            if str(tgt).lower() in str(pred).lower():
                term_to_candidates[src].append(tgt)
            else:
                term_to_candidates[src].append('<MISSING>')

    pseudo_references: dict[str, str] = {}

    for src, candidates in term_to_candidates.items():
        counter = Counter(candidates)
        pseudo_references[src] = counter.most_common(1)[0][0]

    per_term_consistency: dict[str, dict[str, Any]] = {}
    macro_scores: list[float] = []
    weighted_scores: list[float] = []

    for src, candidates in term_to_candidates.items():
        pseudo_ref = pseudo_references[src]
        matches = sum(1 for candidate in candidates if candidate == pseudo_ref)
        consistency = matches / len(candidates) if candidates else 0.0

        per_term_consistency[src] = {
            'occ': len(candidates),
            'pseudo_ref': pseudo_ref,
            'matches': matches,
            'consistency': consistency,
        }

        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))

    return {
        'per_term': per_term_consistency,
        'macro_avg_consistency': (
            sum(macro_scores) / len(macro_scores)
            if macro_scores
            else None
        ),
        'weighted_avg_consistency': (
            sum(weighted_scores) / len(weighted_scores)
            if weighted_scores
            else None
        ),
    }

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
DEFAULT_MODES = ["no_term", "proper_term", "random_term"]
OUTPUT_DIR = RUN_DIR / "outputs_qwen"
SYSTEM_MESSAGE = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."


def detect_target_language(samples: list[dict[str, Any]]) -> tuple[str, str, str, str]:
    for sample in samples:
        if "de" in sample:
            return "de", "German", "deu", "de"
        if "es" in sample:
            return "es", "Spanish", "es", "es"
        if "ru" in sample:
            return "ru", "Russian", "ru", "ru"
    return "de", "German", "deu", "de"


def fmt_metric(value: float | None, digits: int = 2) -> str:
    if value is None:
        return "N/A"
    return f"{value:.{digits}f}"


def term_eval_mode(mode: str) -> str:
    if mode == "proper_term":
        return "proper_term"
    if mode == "random_term":
        return "random_term"
    return "no_term"


def load_model(model_name: str = MODEL_NAME):
    os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    hf_logging.disable_progress_bar()

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return model, tokenizer


def translate_sample(
    model,
    tokenizer,
    sample_en: str,
    terminology: dict[str, str] | None = None,
    target_lang: str = "German",
    max_new_tokens: int = 256,
) -> str:
    prompt = build_translation_prompt(sample_en, target_lang, terminology)
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
    )
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


def run_mode_evaluation(
    mode: str,
    samples: list[dict[str, Any]],
    model,
    tokenizer,
    target_lang: str,
    ref_field: str,
    output_dir: Path,
    data_stem: str,
) -> dict[str, Any]:
    print(f"\n--- Mode: {mode} ---")
    preds: list[str] = []
    records: list[dict[str, Any]] = []

    for sample in tqdm(samples, desc=f"{data_stem} - {mode}"):
        pred = translate_sample(
            model,
            tokenizer,
            sample.get("en", ""),
            terminology=terminology_for_mode(sample, mode),
            target_lang=target_lang,
        )
        preds.append(pred)

        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred)
        records.append(record)

    clean_preds = [strip_output_tags(pred) for pred in preds]
    output_jsonl = output_dir / f"{data_stem}_{mode}_predictions.jsonl"
    save_jsonl(output_jsonl, records)

    refs = [sample.get(ref_field) for sample in samples]
    metrics = compute_bleu_chrf(clean_preds, refs)

    term_mode = term_eval_mode(mode)
    term_acc = terminology_accuracy_advanced(samples, clean_preds, mode=term_mode)
    term_cons = terminology_consistency_advanced(samples, clean_preds, mode=term_mode)

    metrics["terminology_accuracy"] = term_acc
    metrics["terminology_consistency"] = term_cons

    print(f"BLEU: {fmt_metric(metrics['bleu'])}")
    print(f"chrF2++: {fmt_metric(metrics['chrf'])}")
    print(f"Terminology accuracy (ratio %): {fmt_metric(term_acc.get('avg_ratio_pct'))}")
    print(f"Terminology terms counted: {term_acc.get('total_terms', 0)}")
    print(f"Macro-avg consistency: {fmt_metric(term_cons.get('macro_avg_consistency'))}")
    print(f"Weighted-avg consistency: {fmt_metric(term_cons.get('weighted_avg_consistency'))}")

    return {
        "predictions_file": str(output_jsonl),
        "metrics": metrics,
    }

In [ ]:
# Configure run inputs here

HF_TOKEN = os.environ.get("HF_TOKEN", "")  # or paste token string directly
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODES = ["no_term", "proper_term", "random_term"]

# Flat copy mode: notebook + JSONL files are copied together.
# Keep only filename here; resolver checks common runtime directories.
INPUT_FILE = "ende_dev_v1.jsonl"

# Optional: set INPUT_FILE = None to process all *_dev*.jsonl files.
DATA_SEARCH_DIRS = [RUN_DIR, Path("/"), RUN_DIR.parent]
OUTPUT_DIR = RUN_DIR / "outputs_qwen"


def resolve_input_file(input_file: str | os.PathLike[str]) -> Path | None:
    p = Path(input_file)
    if p.is_absolute() and p.exists():
        return p

    candidates: list[Path] = []
    if p.exists():
        candidates.append(p.resolve())

    for base in DATA_SEARCH_DIRS:
        try:
            candidate = (base / p.name).resolve()
            if candidate.exists():
                candidates.append(candidate)
        except Exception:
            continue

    return candidates[0] if candidates else None


if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("INPUT_FILE:", INPUT_FILE)
print("DATA_SEARCH_DIRS:", [str(p) for p in DATA_SEARCH_DIRS])
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODES:", MODES)

In [ ]:
if INPUT_FILE is not None:
    input_path = resolve_input_file(INPUT_FILE)
    if input_path is None:
        raise FileNotFoundError(
            f"INPUT_FILE not found: {INPUT_FILE}. "
            f"Searched in: {[str(p) for p in DATA_SEARCH_DIRS]}"
        )
    files = [str(input_path)]
else:
    all_jsonl: set[Path] = set()
    for base in DATA_SEARCH_DIRS:
        all_jsonl.update(path.resolve() for path in base.glob("*.jsonl"))

    files = sorted(
        str(path)
        for path in all_jsonl
        if "_dev" in path.stem
    )

if not files:
    raise FileNotFoundError(
        f"No input JSONL files found in: {[str(p) for p in DATA_SEARCH_DIRS]}"
    )

print("Found data files:", files)

# Save outputs next to the resolved data file location.
ACTIVE_OUTPUT_DIR = Path(files[0]).parent / "outputs_qwen"
ACTIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("ACTIVE_OUTPUT_DIR:", ACTIVE_OUTPUT_DIR)

model, tokenizer = load_model(MODEL_NAME)
print(f"Model and tokenizer loaded: {MODEL_NAME}")

for filepath in files:
    samples = load_jsonl(filepath)
    _lang_code, lang_name, output_tag, ref_field = detect_target_language(samples)
    data_stem = Path(filepath).stem

    print(
        f"\n=== File: {filepath} | target={lang_name} "
        f"| tag={output_tag} "
        f"| samples={len(samples)} ==="
    )

    all_results: dict[str, Any] = {}
    for mode in MODES:
        all_results[mode] = run_mode_evaluation(
            mode,
            samples,
            model,
            tokenizer,
            lang_name,
            ref_field,
            ACTIVE_OUTPUT_DIR,
            data_stem,
        )

    metrics_path = ACTIVE_OUTPUT_DIR / f"{data_stem}_metrics_summary.json"
    with metrics_path.open("w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print("Saved metrics summary to:", metrics_path)

print("\nEvaluation finished.")